*SVM and Naive Bayes models on Imbalanced dataset*

This notebook presents SVM and Naive Bayes models on Imbalanced dataset.To ensure uniformity, as this is a comparative analysis study,the same data preprocessing and preparation techniques are applied.

In [ ]:
#importing the necessary libraries

#importing the necessary libraries
import pandas as pd
from nltk.tokenize import word_tokenize
import nltk
import re
from nltk.corpus import stopwords
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score

# Download necessary NLTK data
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

Data Cleaning, Label Encoding

**Data cleaning**

This step involves removing  non-alphabetic characters, Converting text to lowercase and removing stop words,Tokenizing and encoding  topic group   into numeric format



In [ ]:

# Loading the dataset
data = pd.read_csv('Service_tickets.csv')

# Getting count of classes
class_counts = data['Topic_group'].value_counts()
print(class_counts)

# Cleaning the text data
def clean_text(text):
    stop_words = set(stopwords.words('english'))
    text = text.lower()  # Convert text to lowercase
    text = re.sub('[^a-zA-Z\s]', '', text)  # Remove non-alphabetic characters
    tokens = word_tokenize(text)  # Tokenize using word_tokenize
    tokens = [word for word in tokens if word not in stop_words]  # Remove stop words
    return ' '.join(tokens)  # Rejoin tokens into a single string

data['Cleaned_Document'] = data['Document'].apply(clean_text)

# Defining the dataframe after cleaning
data = data[['Cleaned_Document', 'Topic_group']]

# Encoding category labels using category codes
data["encoded_label"] = data["Topic_group"].astype('category').cat.codes





Topic_group
Hardware                 13617
HR Support               10915
Access                    7125
Miscellaneous             7060
Storage                   2777
Purchase                  2464
Internal Project          2119
Administrative rights     1760
Name: count, dtype: int64


Data Splitting and TF-IDF Vectorization for Model Training and Evaluation









In [ ]:
# Splitting the data into training (80%) and test (20%) sets
train_data, test_data = train_test_split(data, test_size=0.2, stratify=data['Topic_group'], random_state=42)
X_train = train_data['Cleaned_Document']
X_test = test_data['Cleaned_Document']
y_train = train_data['encoded_label']
y_test = test_data['encoded_label']

# TF-IDF vectorization
tfidf = TfidfVectorizer(max_features=5060)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Converting y_train and y_test to integers after encoding
y_train = y_train.astype(int)
y_test = y_test.astype(int)

# Support Vector Machine (SVM)
svm = SVC()
svm.fit(X_train_tfidf, y_train)
y_pred_svm = svm.predict(X_test_tfidf)
print("Support Vector Machine (SVM) Classification Report:\n", classification_report(y_test, y_pred_svm))

# Naive Bayes
nb = MultinomialNB()
nb.fit(X_train_tfidf, y_train)
y_pred_nb = nb.predict(X_test_tfidf)
print("Naive Bayes Classification Report:\n", classification_report(y_test, y_pred_nb))

Support Vector Machine (SVM) Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.89      0.89      1425
           1       0.87      0.69      0.77       352
           2       0.85      0.85      0.85      2183
           3       0.80      0.88      0.84      2724
           4       0.89      0.83      0.86       424
           5       0.85      0.81      0.83      1412
           6       0.96      0.88      0.92       493
           7       0.92      0.85      0.88       555

    accuracy                           0.85      9568
   macro avg       0.88      0.84      0.86      9568
weighted avg       0.86      0.85      0.85      9568

Naive Bayes Classification Report:
               precision    recall  f1-score   support

           0       0.86      0.79      0.83      1425
           1       0.94      0.28      0.43       352
           2       0.76      0.80      0.78      2183
           3       0.68      0.88      0.77